# 🚀 ACE-Net Master Baseline Model Training & Evaluation
### End-to-End Multimodal Deepfake Consistency Training on 14k Preprocessed Dataset

### 🌟 Workflow Overview:
1. **Load Preprocessed Tensors:** Directly streams `.npy` and `.jpg` features from Google Drive.
2. **Load Pretrained Stage-2 Backbone:** Uses `stage2_acenet.pt` from inyong `checkpoints/` folder.
3. **Train on 14,588 Clips (`final_train_manifest.csv`):** Trains with BCE Loss, AdamW, and Cosine Annealing.
4. **Validate Per Epoch (`final_val_manifest.csv`):** Evaluates Accuracy, AUC, and F1 at every epoch and auto-saves the **`best_baseline_model.pth`** to Google Drive!
5. **Final Testing (`final_test_manifest.csv`):** Produces the official Baseline Results Table for your thesis!

## Step 1: Connect to T4 GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name  :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not enabled! Go to Runtime > Change runtime type > T4 GPU!')

## Step 2: Clone Baseline Repository & Checkout Active Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!git log --oneline -1

## Step 3: Install Core Dependencies

In [ ]:
!pip install -q scikit-learn transformers
print('✅ Dependencies installed successfully!')

## Step 3.5: [GAME-CHANGER] Sync Preprocessed Features to Colab Local NVMe SSD
Bakit ito kailangan?
- Kapag binabasa ang 150k+ files mula sa Google Drive FUSE mount, may 0.3s network delay bawat file -> 140s/it!
- Kapag kinopya sa Local SSD (`/content/preprocessed_local`), nagiging **0.2s/it** na lang!
- **20 Epochs will finish in ~10-15 minutes instead of 16+ hours!**

In [ ]:
import os, time, shutil
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
DRIVE_PREPROCESSED = DRIVE_BASE / 'Baseline preprocessed'
DRIVE_ZIP = DRIVE_BASE / 'baseline_features_all.zip'
LOCAL_PREPROCESSED = Path('/content/preprocessed_local')
LOCAL_PREPROCESSED.mkdir(parents=True, exist_ok=True)

print('=' * 75)
print('🚀 Fast Staging Features to Colab Local NVMe SSD...')
print(f'   Target Local Directory: {LOCAL_PREPROCESSED}')
print('=' * 75)

start_t = time.time()
if DRIVE_ZIP.exists():
    print(f'📦 Found existing master zip in Google Drive: {DRIVE_ZIP}')
    print('⚡ Unzipping directly to local NVMe SSD (takes ~20 seconds)...')
    !unzip -q -o "{DRIVE_ZIP}" -d "{LOCAL_PREPROCESSED}"
else:
    print(f'📁 Syncing directly from Google Drive folder: {DRIVE_PREPROCESSED}')
    print('   (One-time fast copy of 14k preprocessed files, please wait ~8-10 mins)...')
    !cp -r -u "{DRIVE_PREPROCESSED}/." "{LOCAL_PREPROCESSED}/"

elapsed = time.time() - start_t
subdirs = [p.name for p in LOCAL_PREPROCESSED.iterdir() if p.is_dir()]
print(f'\n✅ Staging Complete in {elapsed/60:.2f} mins!')
print(f'   Subfolders verified: {subdirs}')
print('⚡ GPU is now ready for ultra-fast training (~0.2s / batch)!')

## Step 4: Run Stage-2 Baseline Training & Evaluation (1-Click Run)

In [ ]:
import os
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
LOCAL_PREPROCESSED = Path('/content/preprocessed_local')

# Use local SSD if synced, otherwise fallback to Drive
is_complete = (LOCAL_PREPROCESSED / 'TRAIN').exists() and (LOCAL_PREPROCESSED / 'VAL').exists() and (LOCAL_PREPROCESSED / 'TEST').exists()
if is_complete:
    PREPROCESSED_ROOT = LOCAL_PREPROCESSED
    print('⚡ [TURBO MODE] Using Colab Local NVMe SSD features (~0.2s/it)!')
else:
    PREPROCESSED_ROOT = DRIVE_BASE / 'Baseline preprocessed'
    print('⚠️ [STANDARD MODE] Using Google Drive features directly.')

MANIFEST_DIR = Path('/content/Baseline_Training/Manifests/final_manifest_jc')

# Checkpoint paths
STAGE2_CKPT = DRIVE_BASE / 'checkpoints' / 'stage2_acenet.pt'
OUTPUT_MODEL_DIR = DRIVE_BASE / 'checkpoints'

TRAIN_CSV = MANIFEST_DIR / 'final_train_manifest.csv'
VAL_CSV = MANIFEST_DIR / 'final_val_manifest.csv'
TEST_CSV = MANIFEST_DIR / 'final_test_manifest.csv'

print('=' * 75)
print('Train Manifest :', TRAIN_CSV.exists())
print('Val Manifest   :', VAL_CSV.exists())
print('Test Manifest  :', TEST_CSV.exists())
print('Preprocessed   :', PREPROCESSED_ROOT)
print('Stage-2 Ckpt   :', STAGE2_CKPT.exists())
print('=' * 75)

ckpt_arg = f"--ckpt '{STAGE2_CKPT}'" if STAGE2_CKPT.exists() else ""

!python -m src.train_baseline_engine \
    --train-manifest '{TRAIN_CSV}' \
    --val-manifest '{VAL_CSV}' \
    --test-manifest '{TEST_CSV}' \
    --preprocessed-root '{PREPROCESSED_ROOT}' \
    {ckpt_arg} \
    --output-dir '{OUTPUT_MODEL_DIR}' \
    --batch-size 32 \
    --epochs 20 \
    --lr 1e-4 \
    --num-workers 2 \
    --freeze-backbones \
    --device cuda

## Step 5: [AUTOMATIC ARCHIVER] Zip Local Features & Upload to Google Drive
### 📦 Saves `baseline_features_all.zip` to Google Drive for Future 15-Second Instant Loads!

Dahil tapos na ang training at nasa **Local NVMe SSD** na ang lahat ng files:
1. Napakabilis mag-zip sa local NVMe SSD (~1 hanggang 2 minuto lang kumpara sa Drive).
2. Isahang file upload na lang ito (`baseline_features_all.zip`) pabalik sa Google Drive mo.
3. Sa mga susunod na training, **~20 seconds na lang ang unzip** gamit ang Step 3.5!

In [ ]:
import os, time, shutil
from pathlib import Path

LOCAL_PREPROCESSED = Path('/content/preprocessed_local')
DRIVE_DEST_DIR = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
DRIVE_DEST_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DRIVE_ZIP = DRIVE_DEST_DIR / 'baseline_features_all.zip'
LOCAL_TMP_ZIP = Path('/content/baseline_features_all.zip')

if not LOCAL_PREPROCESSED.exists() or not any(LOCAL_PREPROCESSED.iterdir()):
    print('⚠️ Local preprocessed directory is empty or not found.')
elif FINAL_DRIVE_ZIP.exists():
    print(f'ℹ️ Master zip already exists on Google Drive: {FINAL_DRIVE_ZIP}')
    print(f'   Size: {FINAL_DRIVE_ZIP.stat().st_size / (1024**3):.2f} GB')
else:
    subdirs = [p.name for p in LOCAL_PREPROCESSED.iterdir() if p.is_dir()]
    print('=' * 75)
    print('📦 [1/2] Zipping local SSD features to single master archive...')
    print(f'   Folders to zip : {subdirs}')
    print(f'   Target Archive : {LOCAL_TMP_ZIP}')
    print('=' * 75)
    
    start_z = time.time()
    folders_arg = ' '.join(f'"{d}"' for d in subdirs)
    !cd "{LOCAL_PREPROCESSED}" && zip -q -r -1 "{LOCAL_TMP_ZIP}" {folders_arg}
    
    if LOCAL_TMP_ZIP.exists():
        zip_size_gb = LOCAL_TMP_ZIP.stat().st_size / (1024**3)
        z_time = time.time() - start_z
        print(f'\n✅ [1/2] Fast Local Zip Complete! ({zip_size_gb:.2f} GB in {z_time/60:.2f} mins)')
        
        print(f'\n🚀 [2/2] Uploading single master zip to Google Drive ({FINAL_DRIVE_ZIP})...')
        cp_start = time.time()
        shutil.copy2(str(LOCAL_TMP_ZIP), str(FINAL_DRIVE_ZIP))
        cp_time = time.time() - cp_start
        print(f'✅ [2/2] Upload to Google Drive Complete in {cp_time:.1f} seconds!')
        
        # Clean up local temporary zip to free disk space
        LOCAL_TMP_ZIP.unlink()
        
        print('=' * 75)
        print('🎉 SUCCESS: Master dataset archive is now safely stored on your Google Drive!')
        print(f'📍 Saved Path: {FINAL_DRIVE_ZIP}')
        print('=' * 75)
    else:
        print('❌ Failed to create zip archive.')
